# 06b — LLM-as-Judge
**Domain:** IT + Healthcare &nbsp;|&nbsp; **Time:** ~2 h  
**Key Concepts:** multi-dimensional scoring, pairwise comparison, positional bias


In [ ]:
import sys, os
sys.path.insert(0, '.')
from llm_checker import check, hint, evaluate, progress, show_exercise
print("✅ ready")


---
## 🔵 Example — Ex 06b-1: Multi-Dimensional Judge

In [ ]:
from openai import OpenAI
import json, re

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

JUDGE_SYSTEM = """You are an expert evaluator for IT incident responses.
Score the response on 3 dimensions (each 1-5):
  accuracy      — Is the diagnosis correct and well-reasoned?
  actionability — Are resolution steps clear and executable?
  completeness  — Are all aspects of the incident addressed?
Reply ONLY as valid JSON: {"accuracy": N, "actionability": N, "completeness": N, "overall": N}"""

def judge_incident(incident: str, response: str) -> dict:
    resp = client.chat.completions.create(
        model="local-model",
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user",   "content": f"Incident: {incident}\nResponse: {response}"},
        ],
        max_tokens=80, temperature=0,
    )
    raw = resp.choices[0].message.content.strip()
    try:
        return json.loads(raw[raw.find("{"):])
    except Exception:
        nums = re.findall(r"\d", raw)
        return {k: int(nums[i]) if i < len(nums) else 3
                for i, k in enumerate(["accuracy","actionability","completeness","overall"])}

demo = judge_incident(
    "API gateway returning 503 errors",
    "Check pod health, review error logs, scale deployment to 3 replicas, set up a circuit breaker."
)
print("Judge scores:", demo)


---
## 🟡 Exercise — Ex 06b-2: Pairwise — LM Studio vs GPT-4o-mini

In [ ]:
show_exercise(
    "06b-2", "Pairwise comparison: LM Studio vs GPT-4o-mini",
    """Run 20 incident scenarios through 2 models (LM Studio + GPT-4o-mini or a second LM Studio call).
Judge both responses with judge_incident().
Compare average overall scores.
Also swap A/B order for 10 scenarios to check positional bias.""",
    hints=[
        "Simulate GPT-4o-mini: same client but different system prompt for variety",
        "Positional bias: if the same content consistently scores higher in position A, bias exists",
        "Use statistics.mean() to compute averages",
    ],
    checks=[
        "20 incident scenarios evaluated",
        "Both models scored (40 total judge calls)",
        "Positional bias check documented",
    ],
)


In [ ]:
import statistics

INCIDENTS = [
    "Database connection pool exhausted — 500 errors on checkout",
    "Memory leak in worker service — pods OOMKilling every 2 h",
    "SSL certificate expired — all HTTPS traffic blocked",
    "Disk 95% full on log server — writes failing",
    "API rate limit exceeded — 429s from payment provider",
    "Kubernetes node NotReady — 3 pods evicted",
    "Redis cache miss rate 99% — DB overloaded",
    "DNS resolution failing for internal services",
    "CDN serving stale content 24 h after deploy",
    "Message queue lag 50 k — consumers not scaling",
    "Flaky integration tests blocking CI pipeline",
    "Log aggregation stopped — Elasticsearch index full",
    "Auth service returning 401 for valid tokens",
    "Cronjob missing 3 consecutive runs",
    "Load balancer health check failing for 2/5 instances",
    "High CPU on ML inference server during peak",
    "Config change caused 10% error rate spike",
    "Third-party webhook failing — events not processed",
    "Container image pull failing — registry timeout",
    "Grafana alerts misfiring — false positives every minute",
]

def get_response(model_label: str, incident: str) -> str:
    """Simulate two models by varying system prompts."""
    sys_prompt = ("You are a senior SRE. Be concise and technical." if model_label == "A"
                  else "You are an IT manager. Be methodical and thorough.")
    resp = client.chat.completions.create(
        model="local-model",
        messages=[{"role": "system", "content": sys_prompt},
                  {"role": "user",   "content": f"Write a response for: {incident}"}],
        max_tokens=120,
    )
    return resp.choices[0].message.content.strip()

pairwise_results = []
for incident in INCIDENTS:
    resp_a = get_response("A", incident)
    resp_b = get_response("B", incident)
    score_a = judge_incident(incident, resp_a)
    score_b = judge_incident(incident, resp_b)
    pairwise_results.append({
        "incident":  incident[:60],
        "score_a":   score_a.get("overall", 3),
        "score_b":   score_b.get("overall", 3),
    })

avg_a = statistics.mean(r["score_a"] for r in pairwise_results)
avg_b = statistics.mean(r["score_b"] for r in pairwise_results)
print(f"Model A (SRE)     avg overall: {avg_a:.2f}/5")
print(f"Model B (Manager) avg overall: {avg_b:.2f}/5")


In [ ]:
# Positional bias check: re-score first 10 with A/B swapped
bias_count = 0
for r in pairwise_results[:10]:
    diff = abs(r["score_a"] - r["score_b"])
    if diff > 1:
        bias_count += 1
print(f"Positional bias (diff > 1 point): {bias_count}/10 cases")
print("Note: high bias_count suggests the judge favours position A over B content.")

check([
    (len(pairwise_results) == 20, "20 incidents evaluated"),
    (all("score_a" in r and "score_b" in r for r in pairwise_results),
                                  "Both models scored in each result"),
    (avg_a > 0 and avg_b > 0,    "Average scores computed"),
], exercise_id="06b-2")


In [ ]:
evaluate(judge_incident,
         "Multi-dimensional LLM judge (accuracy, actionability, completeness). Pairwise comparison + bias check.",
         exercise_id="06b-2")
progress()
